# MSDM5054 Project 1 - Home Credit Default Risk

Run order: 0 locate data -> 1 write scripts -> 2 quick check -> 3 full run -> 4 interpretability -> 5 robustness -> 6 ablation -> 7 report assets -> 8 download.

Settings: Accelerator = None (CPU is enough). For the long runs use *Save Version -> Save & Run All* so the session is not interrupted.

## 0. Locate the competition data

If nothing is found: right-hand panel -> Input -> **+ Add Input** -> Competitions -> *Home Credit Default Risk*. You must have joined the competition and accepted its rules first.

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root, '->', sorted(files)[:10])

# the pipeline also auto-detects this, but having it explicit makes the cells below readable
DATA = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'application_train.csv' in files:
        DATA = root
        break
print('\nDATA =', DATA)
assert DATA, 'competition data is not attached to this notebook'

## 1. Write the scripts to the working directory

In [ ]:
%%writefile home_credit_pipeline.py
"""
MSDM5054 Project 1 — Home Credit Default Risk: full experimental pipeline
===========================================================================
Research questions
  RQ0  Baselines: logistic regression vs LightGBM (application table only vs all tables)
  RQ1  Is missingness informative (MNAR-like)?  Compare imputation strategies.
  RQ2  Financial inclusion: does the model serve "thin-file" applicants (no Credit Bureau history)
       as well as others, and does alternative data (previous apps / installments / POS / cards) help them more?
  RQ3  Interpretation: SHAP-style contributions (LightGBM pred_contrib, no extra package needed)

Usage
  pip install lightgbm pandas numpy scikit-learn matplotlib
  python home_credit_pipeline.py --data ./data --out ./results            # full run
  python home_credit_pipeline.py --data ./data --out ./results --fast     # 20% sample, quick debug
Recommended: Kaggle Notebook (data already attached, 30GB RAM) or Colab.

Expected files in --data (from the Kaggle competition page):
  application_train.csv application_test.csv bureau.csv bureau_balance.csv previous_application.csv
  POS_CASH_balance.csv installments_payments.csv credit_card_balance.csv
"""
import argparse, gc, json, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[warn] lightgbm not installed: LightGBM experiments will be skipped.")

warnings.filterwarnings("ignore")
SEED = 42
KEY, TARGET = "SK_ID_CURR", "TARGET"


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def reduce_mem(df):
    for c in df.columns:
        if df[c].dtype == "float64":
            df[c] = df[c].astype("float32")
        elif df[c].dtype == "int64" and c not in (KEY, "SK_ID_BUREAU", "SK_ID_PREV"):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


def resolve_data(path):
    """Find the folder that actually contains application_train.csv.

    Handles the common Kaggle cases: the competition data mounted under a
    slightly different folder name, nested one level deeper, or not attached at all.
    """
    if os.path.exists(os.path.join(path, "application_train.csv")):
        return path
    roots = [path, "/kaggle/input", os.getcwd(), "./data"]
    seen = []
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dirpath, _dirs, files in os.walk(r):
            seen.append(dirpath)
            if "application_train.csv" in files:
                log(f"data found at {dirpath} (requested: {path})")
                return dirpath
    listing = "\n  ".join(sorted(set(seen))[:40]) or "(nothing under /kaggle/input)"
    raise SystemExit(
        f"application_train.csv not found. Looked under: {roots}\n"
        f"Directories seen:\n  {listing}\n"
        "Fix: in the notebook editor open the right-hand panel -> Input -> '+ Add Input' -> "
        "Competitions -> 'Home Credit Default Risk' -> Add. You must have clicked "
        "'Join Competition' and accepted the rules first."
    )


def read(data, name, nrows=None):
    return reduce_mem(pd.read_csv(os.path.join(data, name), nrows=nrows))


def num_agg(df, key, prefix, aggs=("mean", "max", "min", "sum")):
    num = df.select_dtypes(include=[np.number]).drop(columns=[c for c in df.columns if c.startswith("SK_ID")], errors="ignore")
    num[key] = df[key]
    g = num.groupby(key).agg(list(aggs))
    g.columns = [f"{prefix}_{a}_{b.upper()}" for a, b in g.columns]
    return g


def cat_share(df, key, prefix, cols):
    """Share of each category level per client (one-hot mean)."""
    if not cols:
        return pd.DataFrame(index=df[key].unique())
    d = pd.get_dummies(df[cols], dummy_na=False).astype("float32")
    d[key] = df[key].values
    g = d.groupby(key).mean()
    g.columns = [f"{prefix}_{c}_SHARE" for c in g.columns]
    return g


# =============================================================== feature engineering
def application(data, nrows=None):
    tr, te = read(data, "application_train.csv", nrows), read(data, "application_test.csv", nrows)
    df = pd.concat([tr, te], ignore_index=True)
    df["N_MISSING_APP"] = df.drop(columns=[TARGET]).isna().sum(axis=1)   # count of missing raw fields
    # known data anomalies
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype("int8")
    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    df.loc[df["CODE_GENDER"] == "XNA", "CODE_GENDER"] = np.nan
    # domain ratios
    df["CREDIT_INCOME"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["ANNUITY_INCOME"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["PAYMENT_RATE"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]            # ~ 1/term
    df["GOODS_CREDIT"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    df["EMPLOYED_AGE"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    df["EXT_MEAN"] = df[ext].mean(axis=1)
    df["EXT_STD"] = df[ext].std(axis=1)
    df["EXT_PROD"] = df[ext].prod(axis=1, min_count=3)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df


def bureau_features(data, nrows=None):
    b, bb = read(data, "bureau.csv", nrows), read(data, "bureau_balance.csv", nrows)
    bb_agg = bb.groupby("SK_ID_BUREAU").agg(BB_MONTHS=("MONTHS_BALANCE", "size"))
    bb_bad = bb.assign(BAD=bb["STATUS"].astype(str).isin(["1", "2", "3", "4", "5"]).astype("float32")).groupby("SK_ID_BUREAU")["BAD"].mean()
    b = b.join(bb_agg, on="SK_ID_BUREAU").join(bb_bad.rename("BB_BAD_SHARE"), on="SK_ID_BUREAU")
    b["DEBT_CREDIT"] = b["AMT_CREDIT_SUM_DEBT"] / b["AMT_CREDIT_SUM"]
    g = num_agg(b, KEY, "BURO", ("mean", "max", "sum"))
    g["BURO_COUNT"] = b.groupby(KEY).size()
    g["BURO_ACTIVE_COUNT"] = b[b["CREDIT_ACTIVE"] == "Active"].groupby(KEY).size()
    g = g.join(cat_share(b, KEY, "BURO", ["CREDIT_ACTIVE", "CREDIT_TYPE"]))
    del b, bb; gc.collect()
    return g


def prev_features(data, nrows=None):
    p = read(data, "previous_application.csv", nrows)
    for c in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
        p.loc[p[c] == 365243, c] = np.nan
    p["APP_CREDIT_RATIO"] = p["AMT_APPLICATION"] / p["AMT_CREDIT"]
    g = num_agg(p, KEY, "PREV", ("mean", "max", "min"))
    g["PREV_COUNT"] = p.groupby(KEY).size()
    g = g.join(cat_share(p, KEY, "PREV", ["NAME_CONTRACT_STATUS", "NAME_CONTRACT_TYPE"]))
    del p; gc.collect()
    return g


def pos_features(data, nrows=None):
    pos = read(data, "POS_CASH_balance.csv", nrows)
    g = num_agg(pos, KEY, "POS", ("mean", "max"))
    g["POS_COUNT"] = pos.groupby(KEY).size()
    del pos; gc.collect()
    return g


def ins_features(data, nrows=None):
    ins = read(data, "installments_payments.csv", nrows)
    ins["PAY_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["PAY_RATIO"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)   # days past due
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)   # days before due
    ins["LATE"] = (ins["DPD"] > 0).astype("float32")
    g = num_agg(ins[[KEY, "PAY_DIFF", "PAY_RATIO", "DPD", "DBD", "LATE", "AMT_PAYMENT"]], KEY, "INS", ("mean", "max", "sum"))
    g["INS_COUNT"] = ins.groupby(KEY).size()
    del ins; gc.collect()
    return g


def cc_features(data, nrows=None):
    cc = read(data, "credit_card_balance.csv", nrows)
    cc["UTIL"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
    g = num_agg(cc, KEY, "CC", ("mean", "max"))
    g["CC_COUNT"] = cc.groupby(KEY).size()
    del cc; gc.collect()
    return g


def build(data, nrows=None, use_aux=True):
    log("application ...")
    df = application(data, nrows)
    if use_aux:
        for name, fn in [("bureau", bureau_features), ("previous", prev_features), ("pos", pos_features),
                         ("installments", ins_features), ("credit card", cc_features)]:
            log(f"{name} ...")
            df = df.join(fn(data, nrows), on=KEY)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    cats = list(df.select_dtypes(exclude=[np.number]).columns)   # version-proof: pandas 3 reads text as StringDtype
    for c in cats:
        df[c] = df[c].astype("category")
    df.columns = [str(c).replace(" ", "_").replace(",", "").replace(":", "").replace('"', "") for c in df.columns]
    tr, te = df[df[TARGET].notna()].reset_index(drop=True), df[df[TARGET].isna()].reset_index(drop=True)
    log(f"features built: train {tr.shape}, test {te.shape}")
    return tr, te


# =============================================================== models
LGB_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=32, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0,
                  n_estimators=5000, random_state=SEED, verbose=-1, n_jobs=-1)


CAT_AS_CODES = False   # set True (or pass --cat-codes) for LightGBM builds that reject pandas categoricals


def _encode_cats_for_lgb(df):
    """Integer-code the categorical columns, returning (frame, column names).

    Some LightGBM builds refuse pandas CategoricalDtype. Passing the codes together with
    `categorical_feature` keeps LightGBM's categorical splits; passing them alone would
    make the trees treat an arbitrary code order as ordinal.
    """
    cats = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if not cats:
        return df, []
    df = df.copy()
    for c in cats:
        df[c] = df[c].astype("category").cat.codes.astype("int32")
    return df, cats


def cv_lgb(X, y, Xte=None, folds=5, params=None, tag="lgb", seed=SEED):
    params = dict(LGB_PARAMS, **(params or {}))
    params["random_state"] = seed
    cat_cols = []
    if CAT_AS_CODES:
        X, cat_cols = _encode_cats_for_lgb(X)
        if Xte is not None:
            Xte, _ = _encode_cats_for_lgb(Xte)
    oof, pred = np.zeros(len(X)), np.zeros(len(Xte)) if Xte is not None else None
    imp, models = pd.Series(0.0, index=X.columns), []
    for k, (a, b) in enumerate(StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y)):
        m = lgb.LGBMClassifier(**params)
        fit_kw = {"categorical_feature": cat_cols} if cat_cols else {}
        m.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)], **fit_kw)
        oof[b] = m.predict_proba(X.iloc[b], num_iteration=m.best_iteration_)[:, 1]
        if Xte is not None:
            pred += m.predict_proba(Xte, num_iteration=m.best_iteration_)[:, 1] / folds
        imp += pd.Series(m.booster_.feature_importance("gain"), index=X.columns) / folds
        models.append(m)
        log(f"  {tag} fold {k}: AUC={roc_auc_score(y[b], oof[b]):.4f}, iters={m.best_iteration_}")
    log(f"{tag} OOF AUC = {roc_auc_score(y, oof):.4f}")
    return oof, pred, imp.sort_values(ascending=False), models


def logit_design(X, add_indicator):
    """One-hot categoricals; numeric -> median impute (+ optional missing indicators) -> standardize."""
    Xn = pd.get_dummies(X, dummy_na=False).astype("float32")
    pipe = make_pipeline(SimpleImputer(strategy="median", add_indicator=add_indicator),
                         StandardScaler(), LogisticRegression(C=0.05, max_iter=3000))
    return Xn, pipe


def cv_logit(X, y, add_indicator=False, folds=5, tag="logit", seed=SEED):
    Xn, pipe = logit_design(X, add_indicator)
    oof = np.zeros(len(X))
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=seed).split(Xn, y):
        pipe.fit(Xn.iloc[a], y[a])
        oof[b] = pipe.predict_proba(Xn.iloc[b])[:, 1]
    log(f"{tag} OOF AUC = {roc_auc_score(y, oof):.4f}")
    return oof


def boot_auc_diff(y, p1, p2, mask=None, B=300, seed=SEED):
    """Bootstrap 95% CI for AUC(p2) - AUC(p1) on subset mask."""
    rng = np.random.default_rng(seed)
    idx = np.where(mask)[0] if mask is not None else np.arange(len(y))
    d = []
    for _ in range(B):
        s = rng.choice(idx, len(idx), replace=True)
        if y[s].min() == y[s].max():
            continue
        d.append(roc_auc_score(y[s], p2[s]) - roc_auc_score(y[s], p1[s]))
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


# =============================================================== experiments
def rq1_missingness(tr, y, out, app_cols):
    """Descriptive: default rate when a column is missing vs observed; predictive: indicators-only model."""
    log("RQ1: missingness analysis")
    X = tr[app_cols]
    rows = []
    for c in X.columns:
        m = X[c].isna().values
        if 0.01 < m.mean() < 0.99:
            rows.append(dict(feature=c, missing_rate=m.mean(), default_if_missing=y[m].mean(),
                             default_if_observed=y[~m].mean(), diff=y[m].mean() - y[~m].mean()))
    tab = pd.DataFrame(rows).sort_values("diff", key=np.abs, ascending=False)
    tab.to_csv(os.path.join(out, "rq1_missing_table.csv"), index=False)
    # indicators-only model: can "which fields are missing" alone predict default?
    M = X.isna().astype("float32")
    M = M.loc[:, (M.mean() > 0.01) & (M.mean() < 0.99)]
    oof = np.zeros(len(M))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(M, y):
        oof[b] = LogisticRegression(max_iter=2000).fit(M.iloc[a], y[a]).predict_proba(M.iloc[b])[:, 1]
    res = {"indicators_only_auc": roc_auc_score(y, oof), "n_indicator_cols": int(M.shape[1])}
    # strategy comparison on application features
    res["logit_median"] = roc_auc_score(y, cv_logit(X, y, False, tag="logit median-impute"))
    res["logit_median_plus_indicator"] = roc_auc_score(y, cv_logit(X, y, True, tag="logit median+indicator"))
    if HAS_LGB:
        res["lgb_native_nan"] = roc_auc_score(y, cv_lgb(X, y, tag="lgb native NaN")[0])
        Xi = X.copy()
        num = Xi.select_dtypes(include=[np.number]).columns
        Xi[num] = Xi[num].fillna(Xi[num].median())
        res["lgb_median_imputed"] = roc_auc_score(y, cv_lgb(Xi, y, tag="lgb median-imputed")[0])
    json.dump(res, open(os.path.join(out, "rq1_results.json"), "w"), indent=2)
    log(f"RQ1 results: {res}")
    return tab, res


def calib_table(y, p, bins=10):
    q = pd.qcut(p, bins, labels=False, duplicates="drop")
    return pd.DataFrame({"pred": p, "obs": y, "bin": q}).groupby("bin").agg(pred=("pred", "mean"), obs=("obs", "mean"), n=("obs", "size"))


def rq2_thinfile(tr, y, p_app, p_full, out):
    log("RQ2: thin-file subgroup analysis")
    thin = tr["BURO_COUNT"].isna().values  # no Credit Bureau record at all
    rows = []
    for name, m in [("all", np.ones(len(y), bool)), ("thin-file (no bureau)", thin), ("with bureau history", ~thin)]:
        r = dict(group=name, n=int(m.sum()), share=float(m.mean()), default_rate=float(y[m].mean()),
                 auc_app_only=roc_auc_score(y[m], p_app[m]), auc_all_tables=roc_auc_score(y[m], p_full[m]),
                 brier_app_only=brier_score_loss(y[m], p_app[m]), brier_all_tables=brier_score_loss(y[m], p_full[m]))
        r["gain_mean"], r["gain_lo"], r["gain_hi"] = boot_auc_diff(y, p_app, p_full, m)
        rows.append(r)
    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(out, "rq2_thinfile.csv"), index=False)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for name, m in [("thin-file", thin), ("with bureau", ~thin)]:
        c = calib_table(y[m], p_full[m])
        ax.plot(c["pred"], c["obs"], "o-", label=name)
    lim = ax.get_xlim()[1]
    ax.plot([0, lim], [0, lim], "k:", lw=.8)
    ax.set_xlabel("mean predicted PD (decile)"); ax.set_ylabel("observed default rate"); ax.legend()
    ax.set_title("Calibration by group (all-tables model, OOF)")
    fig.tight_layout(); fig.savefig(os.path.join(out, "fig_rq2_calibration.png"), dpi=160)
    log("\n" + tab.round(4).to_string(index=False))
    return tab


def rq3_contrib(model, X, out, n=20000):
    log("RQ3: feature contributions (TreeSHAP via pred_contrib)")
    Xs = X.sample(min(n, len(X)), random_state=SEED)
    contrib = model.booster_.predict(Xs, pred_contrib=True)[:, :-1]      # last column = bias
    s = pd.Series(np.abs(contrib).mean(0), index=X.columns).sort_values(ascending=False)
    s.to_csv(os.path.join(out, "rq3_mean_abs_contrib.csv"))
    top = s.head(20)[::-1]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.barh(top.index, top.values); ax.set_xlabel("mean |contribution| (log-odds)")
    fig.tight_layout(); fig.savefig(os.path.join(out, "fig_rq3_top20.png"), dpi=160)
    return s


# =============================================================== main
def app_feature_cols(data, feats):
    raw_app = pd.read_csv(os.path.join(data, "application_train.csv"), nrows=5).columns
    return [c for c in feats if c in raw_app or c in
            ("DAYS_EMPLOYED_ANOM", "CREDIT_INCOME", "ANNUITY_INCOME", "PAYMENT_RATE", "GOODS_CREDIT",
             "EMPLOYED_AGE", "INCOME_PER_PERSON", "EXT_MEAN", "EXT_STD", "EXT_PROD", "N_MISSING_APP")]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="./data")
    ap.add_argument("--out", default="./results")
    ap.add_argument("--fast", action="store_true", help="20%% row sample of application tables for debugging")
    ap.add_argument("--nrows", type=int, default=None, help="read only first n rows of every csv (smoke test)")
    ap.add_argument("--cat-codes", dest="cat_codes", action="store_true",
                    help="pass categoricals to LightGBM as integer codes (for builds that reject pandas categoricals)")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    global CAT_AS_CODES
    CAT_AS_CODES = args.cat_codes

    args.data = resolve_data(args.data)
    tr, te = build(args.data, args.nrows, use_aux=True)
    if args.fast:
        tr = tr.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
    y = tr[TARGET].astype(int).values
    feats = [c for c in tr.columns if c not in (KEY, TARGET)]
    app_cols = app_feature_cols(args.data, feats)
    summary = {"n_train": int(len(tr)), "default_rate": float(y.mean()), "n_features_all": len(feats), "n_features_app": len(app_cols)}

    # ---------- RQ0 baselines
    p_logit = cv_logit(tr[app_cols], y, add_indicator=True, tag="RQ0 logit (app only)")
    summary["auc_logit_app"] = roc_auc_score(y, p_logit)
    if HAS_LGB:
        p_app, _, imp_app, _ = cv_lgb(tr[app_cols], y, tag="RQ0 lgb (app only)")
        p_full, p_test, imp_full, models = cv_lgb(tr[feats], y, te[feats], tag="RQ0 lgb (all tables)")
        summary["auc_lgb_app"], summary["auc_lgb_all"] = roc_auc_score(y, p_app), roc_auc_score(y, p_full)
        imp_full.to_csv(os.path.join(args.out, "lgb_gain_importance.csv"))
        if not args.fast:
            pd.DataFrame({KEY: te[KEY].astype(int), TARGET: p_test}).to_csv(os.path.join(args.out, "submission.csv"), index=False)
            log("submission.csv written -> upload to Kaggle (Late Submission)")
        # ---------- RQ2, RQ3
        rq2_thinfile(tr, y, p_app, p_full, args.out)
        rq3_contrib(models[0], tr[feats], args.out)
    # ---------- RQ1
    rq1_missingness(tr, y, args.out, app_cols)

    json.dump(summary, open(os.path.join(args.out, "summary.json"), "w"), indent=2)
    log(f"SUMMARY {summary}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_interpretability.py
"""
MSDM5054 Project 1 — Direction 2: Interpretability (SHAP) & publication figures
================================================================================
Retrains the production all-tables LightGBM (5-fold CV, seed=42, identical to the
main pipeline) on the real competition data and produces every artifact needed for
the Direction-2 deliverables:

  1. OOF predictions of the all-tables model  -> fig_rq2_calibration (redrawn)
  2. TreeSHAP contributions on a fixed sample -> fig_rq3_top20 (redrawn),
                                                 SHAP dependence plots (top-5),
                                                 group-wise SHAP comparison
  3. Group masks (thin-file vs with bureau)   -> RQ2 / RQ3 cross-analysis

Artifacts written to --out (default ./results):
  oof_alltables.npy          OOF predicted probabilities (n_train,)
  y.npy, thin_mask.npy       labels and thin-file mask (BURO_COUNT is NaN)
  shap_contrib.npy           (n_sample, n_features) TreeSHAP values, fold-0 model
  shap_sample_values.npy     matching feature values (same rows)
  shap_sample_cols.json      feature names
  shap_group_contrib.csv     mean |SHAP| per group (thin / with bureau / all)
  shap_mean_abs_contrib.csv  overall mean |SHAP| (sanity check vs report)
  interpretability_summary.json

Usage
  python run_interpretability.py --data ./data --out ./results
"""
import argparse, json, os, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# reuse the production pipeline unchanged
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))  # import the pipeline from this folder
from home_credit_pipeline import build, cv_lgb, resolve_data, SEED, KEY, TARGET  # noqa: E402

N_SHAP_SAMPLE = 50000


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="./data")
    ap.add_argument("--out", default="./results")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    args.data = resolve_data(args.data)
    log("building features (7 tables) ...")
    tr, _te = build(args.data, use_aux=True)
    # pandas >= 3.0 uses the new 'str' dtype, which the original pipeline's
    # `dtype == "object"` check misses — convert every non-numeric column to
    # category explicitly so LightGBM accepts the frame.
    for c in tr.columns:
        if c not in (KEY, TARGET) and not pd.api.types.is_numeric_dtype(tr[c]):
            tr[c] = tr[c].astype("category")
    y = tr[TARGET].astype(int).values
    feats = [c for c in tr.columns if c not in (KEY, TARGET)]
    thin = tr["BURO_COUNT"].isna().values

    # ---------- 1. all-tables model, same config/seed as the main pipeline
    log("training all-tables LightGBM (5-fold, seed=42) ...")
    p_full, _pred_te, _imp, models = cv_lgb(tr[feats], y, tag="D2 all-tables", seed=SEED)
    auc_full = roc_auc_score(y, p_full)
    log(f"all-tables OOF AUC = {auc_full:.5f}  (report: 0.79261)")
    np.save(os.path.join(args.out, "oof_alltables.npy"), p_full.astype("float64"))
    np.save(os.path.join(args.out, "y.npy"), y)
    np.save(os.path.join(args.out, "thin_mask.npy"), thin)

    # per-group AUC (sanity check vs rq2_thinfile.csv)
    for name, m in [("thin", thin), ("buro", ~thin)]:
        log(f"  AUC[{name}] = {roc_auc_score(y[m], p_full[m]):.5f}")

    # ---------- 2. TreeSHAP on a fixed sample, fold-0 model (same recipe as rq3_contrib)
    log(f"computing TreeSHAP on {N_SHAP_SAMPLE:,} rows (fold-0 model) ...")
    Xs = tr[feats].sample(min(N_SHAP_SAMPLE, len(tr)), random_state=SEED)
    contrib = models[0].booster_.predict(Xs, pred_contrib=True)[:, :-1]  # drop bias column
    np.save(os.path.join(args.out, "shap_contrib.npy"), contrib.astype("float32"))
    # categorical columns -> integer codes (NaN preserved), so everything fits float32
    Xn = Xs.copy()
    for c in Xn.columns:
        if str(Xn[c].dtype) in ("category", "str", "object"):
            codes = Xn[c].astype("category").cat.codes
            Xn[c] = codes.astype("float32").mask(codes < 0, np.nan)
    np.save(os.path.join(args.out, "shap_sample_values.npy"), Xn.to_numpy(dtype="float32"))
    json.dump(list(Xs.columns), open(os.path.join(args.out, "shap_sample_cols.json"), "w"))

    mean_abs = pd.Series(np.abs(contrib).mean(0), index=feats).sort_values(ascending=False)
    mean_abs.to_csv(os.path.join(args.out, "shap_mean_abs_contrib.csv"))

    # ---------- 3. group-wise mean |SHAP| (thin-file vs with bureau)
    thin_pos = tr.index.get_indexer(Xs.index)  # Xs sampled from tr, so all found
    m_thin = thin[thin_pos]
    g = pd.DataFrame({
        "all": np.abs(contrib).mean(0),
        "thin_file": np.abs(contrib[m_thin]).mean(0),
        "with_bureau": np.abs(contrib[~m_thin]).mean(0),
    }, index=feats)
    g["diff_thin_minus_buro"] = g["thin_file"] - g["with_bureau"]
    g.sort_values("all", ascending=False).to_csv(os.path.join(args.out, "shap_group_contrib.csv"))
    log(f"group sample sizes: thin={int(m_thin.sum()):,}, with_bureau={int((~m_thin).sum()):,}")

    # sanity check against the report's rq3_mean_abs_contrib.csv
    ref = "../results/rq3_mean_abs_contrib.csv"
    if os.path.exists(ref):
        old = pd.read_csv(ref, index_col=0).iloc[:, 0]
        cmp = pd.DataFrame({"report": old, "retrained": mean_abs}).dropna()
        corr = np.corrcoef(cmp["report"], cmp["retrained"])[0, 1]
        log(f"sanity: corr(report |SHAP|, retrained |SHAP|) = {corr:.4f} over {len(cmp)} features")

    summary = {
        "auc_all_tables": auc_full,
        "auc_thin": float(roc_auc_score(y[thin], p_full[thin])),
        "auc_buro": float(roc_auc_score(y[~thin], p_full[~thin])),
        "n_shap_sample": int(len(Xs)),
        "n_thin_sample": int(m_thin.sum()),
        "top10_mean_abs_shap": {k: float(v) for k, v in mean_abs.head(10).items()},
    }
    json.dump(summary, open(os.path.join(args.out, "interpretability_summary.json"), "w"), indent=2)
    log(f"SUMMARY {summary}")
    log("DONE")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_robustness.py
"""
MSDM5054 Project 1 — Direction 1: Robustness & hyperparameter neighbourhood check
=================================================================================
Run multi-seed stability checks and a small hyperparameter grid on top of the
already-built feature matrix. Feature engineering is NOT repeated per seed —
only model fitting and fold assignment vary, which is exactly the source of
variance we want to quantify.

Outputs (in --out):
  robustness_multiseed.csv   one row per seed: OOF AUC for app-only / all-tables,
                             RQ2 thin-file & with-bureau gains (mean, lo, hi),
                             RQ1 logit median vs +indicator AUC.
  robustness_tuning.csv       one row per hyperparameter setting (3-fold CV AUC
                             on the all-tables design).

Usage
  python run_robustness.py --data ./data --out ./results
  python run_robustness.py --data ./data --out ./results --fast     # 20% sample, smoke test
  python run_robustness.py --data ./data --out ./results --seeds 42,2026,7,123,2024

Time budget (Kaggle Notebook, 30 GB RAM):
  - feature engineering once:            ~3-5 min
  - all-tables 5-fold CV per seed:      ~15-20 min  x 5 seeds = ~1.5-2 h
  - app-only 5-fold CV per seed:        ~3-5 min   x 5 seeds = ~25 min
  - logit strategies per seed:          ~2 min     x 5 seeds = ~10 min
  - tuning grid (3-fold, 6 settings):  ~10 min    x 6      = ~1 h
  TOTAL ~ 4 h.  Use --fast for a quick sanity run.
"""
import argparse, json, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from home_credit_pipeline import (
    build, cv_lgb, cv_logit, boot_auc_diff, log, resolve_data, SEED, KEY, TARGET, HAS_LGB,
)


# Seeds used for the multi-seed stability run. 5 seeds give df=4 for the std,
# which is the minimum that makes a reported std interpretable.
DEFAULT_SEEDS = [42, 2026, 7, 123, 2024]

# Hyperparameter neighbourhood grid. 3-fold CV per row to keep it fast.
# "current" reproduces the production setting exactly.
TUNING_GRID = [
    dict(tag="current",        num_leaves=32, min_child_samples=100, colsample_bytree=0.5, reg_lambda=5.0),
    dict(tag="deeper",         num_leaves=63, min_child_samples=100, colsample_bytree=0.5, reg_lambda=5.0),
    dict(tag="more_conserv",   num_leaves=15, min_child_samples=200, colsample_bytree=0.5, reg_lambda=5.0),
    dict(tag="colsample_0.8",  num_leaves=32, min_child_samples=100, colsample_bytree=0.8, reg_lambda=5.0),
    dict(tag="reg_l20",        num_leaves=32, min_child_samples=100, colsample_bytree=0.5, reg_lambda=20.0),
    dict(tag="kaggle_common",  num_leaves=31, min_child_samples=30,  colsample_bytree=0.7, reg_lambda=1.0),
]


def thin_file_mask(tr):
    """Thin-file = no Credit Bureau record at all (BURO_COUNT is NaN after left join)."""
    return tr["BURO_COUNT"].isna().values


def run_multiseed(tr, y, app_cols, feats, seeds, out):
    log(f"=== Multi-seed robustness: {len(seeds)} seeds ===")
    thin = thin_file_mask(tr)
    rows = []
    for seed in seeds:
        log(f"--- seed = {seed} ---")
        # app-only model
        p_app, _, _, _ = cv_lgb(tr[app_cols], y, tag=f"seed{seed} app-only", seed=seed)
        auc_app = roc_auc_score(y, p_app)
        # all-tables model
        p_full, _, _, _ = cv_lgb(tr[feats], y, tag=f"seed{seed} all-tables", seed=seed)
        auc_full = roc_auc_score(y, p_full)
        # RQ2 subgroup gains with bootstrap CI (re-computed per seed because OOF changes)
        g_thin = boot_auc_diff(y, p_app, p_full, thin, seed=seed)
        g_buro = boot_auc_diff(y, p_app, p_full, ~thin, seed=seed)
        # RQ1: logit median vs +indicator on app features
        auc_logit_med = roc_auc_score(y, cv_logit(tr[app_cols], y, add_indicator=False,
                                                   tag=f"seed{seed} logit median", seed=seed))
        auc_logit_ind = roc_auc_score(y, cv_logit(tr[app_cols], y, add_indicator=True,
                                                   tag=f"seed{seed} logit median+ind", seed=seed))
        rows.append(dict(
            seed=seed,
            auc_lgb_app=auc_app,
            auc_lgb_all=auc_full,
            rq2_gain_thin_mean=g_thin[0], rq2_gain_thin_lo=g_thin[1], rq2_gain_thin_hi=g_thin[2],
            rq2_gain_buro_mean=g_buro[0], rq2_gain_buro_lo=g_buro[1], rq2_gain_buro_hi=g_buro[2],
            rq1_logit_median=auc_logit_med,
            rq1_logit_plus_ind=auc_logit_ind,
            rq1_ind_boost=auc_logit_ind - auc_logit_med,
        ))
        log(f"  seed {seed}: app={auc_app:.4f}  all={auc_full:.4f}  "
            f"gain_thin={g_thin[0]:+.4f} [{g_thin[1]:+.4f},{g_thin[2]:+.4f}]  "
            f"gain_buro={g_buro[0]:+.4f} [{g_buro[1]:+.4f},{g_buro[2]:+.4f}]  "
            f"RQ1 boost={auc_logit_ind - auc_logit_med:+.4f}")
    df = pd.DataFrame(rows)
    # append a summary row with mean / std across seeds
    summary = {c: ("mean", "std") for c in df.columns if c != "seed"}
    agg = df.agg(summary)
    log("=== Across-seed summary ===")
    log(agg.round(4).to_string())
    df.to_csv(os.path.join(out, "robustness_multiseed.csv"), index=False)
    agg.to_csv(os.path.join(out, "robustness_multiseed_summary.csv"))
    return df


def run_tuning(tr, y, feats, out, base_seed=SEED):
    log("=== Hyperparameter neighbourhood check (3-fold CV) ===")
    rows = []
    for setting in TUNING_GRID:
        tag = setting.pop("tag")
        log(f"--- {tag}: {setting} ---")
        # 3-fold CV, fixed seed, no test prediction — we only care about OOF AUC here.
        oof, _, _, _ = cv_lgb(tr[feats], y, folds=3, params=setting,
                              tag=f"tune {tag}", seed=base_seed)
        auc = roc_auc_score(y, oof)
        rows.append(dict(tag=tag, **setting, oof_auc_3fold=auc))
        log(f"  {tag}: OOF AUC (3-fold) = {auc:.4f}")
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, "robustness_tuning.csv"), index=False)
    log("=== Tuning summary ===")
    log(df.round(4).to_string(index=False))
    return df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="./data")
    ap.add_argument("--out", default="./results")
    ap.add_argument("--fast", action="store_true", help="20%% row sample for smoke testing")
    ap.add_argument("--seeds", type=str, default=",".join(map(str, DEFAULT_SEEDS)),
                    help="comma-separated list of seeds, e.g. 42,2026,7")
    ap.add_argument("--skip-tuning", action="store_true", help="only run multi-seed, skip hyperparam grid")
    ap.add_argument("--skip-multiseed", action="store_true", help="only run tuning grid")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    if not HAS_LGB:
        raise SystemExit("lightgbm is required for run_robustness.py")

    args.data = resolve_data(args.data)
    seeds = [int(s) for s in args.seeds.split(",") if s.strip()]

    # Feature engineering runs ONCE — it has no randomness.
    tr, _ = build(args.data, use_aux=True)
    if args.fast:
        tr = tr.sample(frac=0.2, random_state=SEED).reset_index(drop=True)
    y = tr[TARGET].astype(int).values
    feats = [c for c in tr.columns if c not in (KEY, TARGET)]
    # Re-derive app_cols the same way the main pipeline does.
    raw_app = pd.read_csv(os.path.join(args.data, "application_train.csv"), nrows=5).columns
    app_cols = [c for c in feats if c in raw_app or c in
                ("DAYS_EMPLOYED_ANOM", "CREDIT_INCOME", "ANNUITY_INCOME", "PAYMENT_RATE", "GOODS_CREDIT",
                 "EMPLOYED_AGE", "INCOME_PER_PERSON", "EXT_MEAN", "EXT_STD", "EXT_PROD", "N_MISSING_APP")]
    log(f"train={tr.shape[0]}, n_features={len(feats)}, n_app_features={len(app_cols)}, seeds={seeds}")

    if not args.skip_multiseed:
        run_multiseed(tr, y, app_cols, feats, seeds, args.out)
    if not args.skip_tuning:
        run_tuning(tr, y, feats, args.out)

    log("DONE. Files written:")
    for f in ["robustness_multiseed.csv", "robustness_multiseed_summary.csv", "robustness_tuning.csv"]:
        p = os.path.join(args.out, f)
        if os.path.exists(p):
            log(f"  {p}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_ablation.py
from pathlib import Path
import argparse
import sys

import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score


PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from home_credit_pipeline import build, cv_lgb, KEY, TARGET, SEED


SOURCE_PREFIXES = {
    "Bureau": ("BURO_",),
    "Previous applications": ("PREV_",),
    "POS/CASH": ("POS_",),
    "Installments": ("INS_",),
    "Credit cards": ("CC_",),
}


def select_features(features, remove_prefixes=()):
    return [
        c
        for c in features
        if not any(c.startswith(prefix) for prefix in remove_prefixes)
    ]


def run_model(tr, y, features, tag):
    oof, _, _, _ = cv_lgb(
        tr[features],
        y,
        Xte=None,
        folds=5,
        tag=tag,
    )
    return roc_auc_score(y, oof)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="./data")
    parser.add_argument("--out", default="./direction4/results")
    parser.add_argument(
        "--figures",
        default="./direction4/figures",
    )
    parser.add_argument(
        "--nrows",
        type=int,
        default=None,
        help="Read only first n rows from each CSV for debugging.",
    )
    parser.add_argument(
        "--fast",
        action="store_true",
        help="Use 20%% of the training rows for debugging.",
    )
    args = parser.parse_args()

    out_dir = Path(args.out)
    fig_dir = Path(args.figures)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 70)
    print("Direction 4: Alternative-data source ablation")
    print("=" * 70)
    print(f"Data: {args.data}")
    print(f"Seed: {SEED}")

    print("\nBuilding full feature matrix...")
    tr, _ = build(args.data, args.nrows, use_aux=True)

    if args.fast:
        tr = (
            tr.sample(frac=0.2, random_state=SEED)
            .reset_index(drop=True)
        )
        print(f"Fast mode: using {len(tr):,} training rows")

    y = tr[TARGET].astype(int).values
    features = [c for c in tr.columns if c not in (KEY, TARGET)]

    print(f"Training rows: {len(tr):,}")
    print(f"Total features: {len(features)}")

    results = []

    print("\n[1/6] Full model")
    auc_full = run_model(
        tr,
        y,
        features,
        "ablation full",
    )
    results.append(
        {
            "removed_source": "None",
            "n_features": len(features),
            "oof_auc": auc_full,
            "auc_drop_vs_full": 0.0,
        }
    )
    print(f"Full model OOF AUC = {auc_full:.6f}")

    for i, (source, prefixes) in enumerate(SOURCE_PREFIXES.items(), start=2):
        selected = select_features(features, prefixes)

        print(f"\n[{i}/6] Remove: {source}")
        print(f"Remaining features: {len(selected)}")

        auc = run_model(
            tr,
            y,
            selected,
            f"ablation - {source}",
        )

        drop = auc_full - auc

        results.append(
            {
                "removed_source": source,
                "n_features": len(selected),
                "oof_auc": auc,
                "auc_drop_vs_full": drop,
            }
        )

        print(f"AUC = {auc:.6f}")
        print(f"AUC drop = {drop:+.6f}")

    result_df = pd.DataFrame(results)

    csv_path = out_dir / "ablation_auc.csv"
    result_df.to_csv(csv_path, index=False)

    print("\n" + "=" * 70)
    print("Ablation results")
    print("=" * 70)
    print(result_df.to_string(index=False))
    print(f"\nSaved: {csv_path}")

    plot_df = result_df[result_df["removed_source"] != "None"].copy()
    plot_df = plot_df.sort_values("auc_drop_vs_full")

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.barh(
        plot_df["removed_source"],
        plot_df["auc_drop_vs_full"],
    )
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel("OOF AUC drop relative to full model")
    ax.set_ylabel("Removed data source")
    ax.set_title("Predictive contribution of alternative data sources")
    ax.grid(axis="x", alpha=0.2)

    for y_pos, value in enumerate(plot_df["auc_drop_vs_full"]):
        ax.text(
            value,
            y_pos,
            f" {value:+.4f}",
            va="center",
        )

    fig.tight_layout()

    fig_path = fig_dir / "fig_ablation_auc.png"
    fig.savefig(fig_path, dpi=300)
    plt.close(fig)

    print(f"Saved: {fig_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_report_assets.py
"""
Turn pipeline outputs (results/) into LaTeX tables + figures used by report_main.tex.
Every number in the report then comes straight from the result files (no manual copying).

Usage:
    python make_report_assets.py --results ../results --kaggle_public 0.xxx --kaggle_private 0.xxx
Outputs (in this folder):
    tables/tab_main.tex  tables/tab_missing.tex  tables/tab_thinfile.tex  tables/macros.tex
    fig_rq1_missing.png  (+ copies of fig_rq2_calibration.png, fig_rq3_top20.png if present)
Missing results are rendered as "TBD" so the LaTeX always compiles.
"""
import argparse, json, os, shutil
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ap = argparse.ArgumentParser()
ap.add_argument("--results", default="../results")
ap.add_argument("--kaggle_public", default=None)
ap.add_argument("--kaggle_private", default=None)
ap.add_argument("--keep_figures", action="store_true", help="do not regenerate/overwrite the polished figures")
a = ap.parse_args()
R = a.results
os.makedirs("tables", exist_ok=True)


def load_json(name):
    p = os.path.join(R, name)
    return json.load(open(p)) if os.path.exists(p) else {}


def f(x, d=4):
    return "TBD" if x is None else (f"{x:.{d}f}" if isinstance(x, (int, float)) else str(x))


def esc(s):
    return str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")


S, R1 = load_json("summary.json"), load_json("rq1_results.json")

# ---------------- robustness (Direction 1)
rp = os.path.join(R, "robustness_summary.csv")
rows_rb, rb_macros = [], {}
if os.path.exists(rp):
    rb = pd.read_csv(rp, index_col=0)
    seeds = pd.read_csv(os.path.join(R, "robustness_seeds.csv"))
    labels = [("auc_app", "OOF AUC, application only"), ("auc_all", "OOF AUC, all tables"),
              ("auc_all_thin", "AUC, thin-file (all tables)"), ("auc_all_thick", "AUC, with bureau (all tables)"),
              ("gain_thin", "Auxiliary-data gain, thin-file"), ("gain_thick", "Auxiliary-data gain, with bureau"),
              ("gain_diff", "Difference of the two gains")]
    for k, lab in labels:
        if k in rb.index:
            rows_rb.append((lab, rb.loc[k, "mean"], rb.loc[k, "std"], rb.loc[k, "min"], rb.loc[k, "max"]))
    rb_macros = {"RobSeeds": str(len(seeds)),
                 "RobAucAll": f(rb.loc["auc_all", "mean"]), "RobAucAllSd": f(rb.loc["auc_all", "std"]),
                 "RobGainThin": f(rb.loc["gain_thin", "mean"]), "RobGainThick": f(rb.loc["gain_thick", "mean"]),
                 "RobGainDiff": f(rb.loc["gain_diff", "mean"]), "RobGainDiffSd": f(rb.loc["gain_diff", "std"]),
                 "RobHolds": str(int((seeds.gain_diff > 0).sum()))}
else:
    rb_macros = {k: "TBD" for k in ["RobSeeds", "RobAucAll", "RobAucAllSd", "RobGainThin",
                                    "RobGainThick", "RobGainDiff", "RobGainDiffSd", "RobHolds"]}
with open("tables/tab_robust.tex", "w") as fh:
    fh.write("\\begin{tabular}{lcccc}\n\\toprule\nQuantity & mean & sd & min & max\\\\\n\\midrule\n")
    if rows_rb:
        for lab, m, sd, lo, hi in rows_rb:
            fh.write(f"{lab} & {m:.4f} & {sd:.4f} & {lo:.4f} & {hi:.4f}\\\\\n")
    else:
        for _, lab in [("", "OOF AUC, all tables"), ("", "Auxiliary-data gain, thin-file"),
                       ("", "Auxiliary-data gain, with bureau"), ("", "Difference of the two gains")]:
            fh.write(f"{lab} & TBD & TBD & TBD & TBD\\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n")

# ---------------- macros (numbers quoted in the text)
macros = {
    "NTrain": f"{S['n_train']:,}" if "n_train" in S else "TBD",
    "DefaultRate": f(100 * S["default_rate"], 2) + r"\%" if "default_rate" in S else "TBD",
    "NFeatAll": f(S.get("n_features_all"), 0) if "n_features_all" in S else "TBD",
    "NFeatApp": f(S.get("n_features_app"), 0) if "n_features_app" in S else "TBD",
    "AucLogit": f(S.get("auc_logit_app")), "AucLgbApp": f(S.get("auc_lgb_app")), "AucLgbAll": f(S.get("auc_lgb_all")),
    "KagglePublic": a.kaggle_public or "TBD", "KagglePrivate": a.kaggle_private or "TBD",
    "AucIndOnly": f(R1.get("indicators_only_auc")),
}
thin_path = os.path.join(R, "rq2_thinfile.csv")
if os.path.exists(thin_path):
    t = pd.read_csv(thin_path)
    thin = t[t.group.str.startswith("thin")].iloc[0]
    macros["ThinShare"] = f(100 * thin.share, 1) + r"\%"
else:
    macros["ThinShare"] = "TBD"
macros.update(rb_macros)
with open("tables/macros.tex", "w") as fh:
    for k, v in macros.items():
        fh.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")

# ---------------- Table 1
rows = [("Logistic regression (L2)", "application", S.get("auc_logit_app"), "--", "--"),
        ("LightGBM", "application", S.get("auc_lgb_app"), "--", "--"),
        ("LightGBM", "all tables", S.get("auc_lgb_all"), a.kaggle_public or "TBD", a.kaggle_private or "TBD")]
with open("tables/tab_main.tex", "w") as fh:
    fh.write("\\begin{tabular}{llccc}\n\\toprule\nModel & Features & OOF AUC & Kaggle public & Kaggle private\\\\\n\\midrule\n")
    for r in rows:
        fh.write(f"{r[0]} & {r[1]} & {f(r[2])} & {r[3]} & {r[4]}\\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n")

# ---------------- Table 2
rows = [("Missingness indicators only (logistic)", R1.get("indicators_only_auc")),
        ("Logistic, median imputation", R1.get("logit_median")),
        ("Logistic, median imputation + indicators", R1.get("logit_median_plus_indicator")),
        ("LightGBM, native NaN handling", R1.get("lgb_native_nan")),
        ("LightGBM, median-imputed", R1.get("lgb_median_imputed"))]
with open("tables/tab_missing.tex", "w") as fh:
    fh.write("\\begin{tabular}{lc}\n\\toprule\nStrategy (application features) & OOF AUC\\\\\n\\midrule\n")
    for n, v in rows:
        fh.write(f"{n} & {f(v)}\\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n")

# ---------------- Table 3
with open("tables/tab_thinfile.tex", "w") as fh:
    fh.write("\\begin{tabular}{lrcccccc}\n\\toprule\nGroup & $n$ & Default & AUC (app) & AUC (all) & Gain [95\\% CI] & Brier (all)\\\\\n\\midrule\n")
    if os.path.exists(thin_path):
        for _, r in t.iterrows():
            fh.write(f"{esc(r.group)} & {int(r.n):,} & {100*r.default_rate:.2f}\\% & {r.auc_app_only:.4f} & {r.auc_all_tables:.4f} & "
                     f"{r.gain_mean:+.4f} [{r.gain_lo:+.4f}, {r.gain_hi:+.4f}] & {r.brier_all_tables:.4f}\\\\\n")
    else:
        for g in ["All", "Thin-file (no bureau)", "With bureau history"]:
            fh.write(f"{g} & TBD & TBD & TBD & TBD & TBD & TBD\\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n")

# ---------------- Figure 1 (RQ1)
mp = os.path.join(R, "rq1_missing_table.csv")
if os.path.exists(mp) and not a.keep_figures:
    m = pd.read_csv(mp)
    # Many columns share an identical missingness pattern (e.g. the six AMT_REQ_CREDIT_BUREAU_*
    # fields, or ENTRANCES_{AVG,MODE,MEDI}); collapse them so the figure shows distinct patterns.
    m["_key"] = m.missing_rate.round(6).astype(str) + "_" + m.default_if_missing.round(6).astype(str)
    grp = m.groupby("_key", sort=False)
    m = grp.first().assign(n_same=grp.size().values).reset_index(drop=True).head(10)[::-1]
    labels = [f"{c} ({100*r:.0f}% miss)" + (f" +{k-1} alike" if k > 1 else "")
              for c, r, k in zip(m.feature, m.missing_rate, m.n_same)]
    fig, ax = plt.subplots(figsize=(6.4, 4))
    y = range(len(m))
    ax.barh([i + 0.2 for i in y], 100 * m.default_if_missing, height=0.4, label="field missing")
    ax.barh([i - 0.2 for i in y], 100 * m.default_if_observed, height=0.4, label="field observed")
    ax.axvline(100 * 0.0807, color="k", ls=":", lw=1, label="overall rate (8.1%)")
    ax.set_yticks(list(y)); ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("default rate (%)"); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, 1.13), ncol=3, frameon=False); ax.grid(axis="x", alpha=.3)
    fig.tight_layout(); fig.savefig("fig_rq1_missing.png", dpi=180)
for fn in ([] if a.keep_figures else ["fig_rq2_calibration.png", "fig_rq3_top20.png"]):
    if os.path.exists(os.path.join(R, fn)):
        shutil.copy(os.path.join(R, fn), fn)
print("assets written:", sorted(os.listdir("tables")), [x for x in os.listdir(".") if x.endswith(".png")])


## 2. Quick check (20% row sample, a few minutes)

Stop here if the final `SUMMARY {...}` line does not appear.

In [ ]:
!python home_credit_pipeline.py --data {DATA} --out ./results_fast --fast

## 3. Full run (~40-70 min)

Writes `results/submission.csv` plus every RQ1-RQ3 artefact.

In [ ]:
!python home_credit_pipeline.py --data {DATA} --out ./results

In [ ]:
import json, pandas as pd

print(json.load(open('results/summary.json')))
print(json.load(open('results/rq1_results.json')))
pd.read_csv('results/rq2_thinfile.csv')

## 4. Interpretability: group-wise SHAP (~15 min)

In [ ]:
!python run_interpretability.py --data {DATA} --out ./results

## 5. Robustness: 5 seeds + hyperparameter neighbourhood (~2-3 h)

Add `--cat-codes` only if this LightGBM build rejects pandas categorical columns.

In [ ]:
!python run_robustness.py --data {DATA} --out ./results

## 6. Leave-one-source-out ablation (~3 h: five full refits)

In [ ]:
!python run_ablation.py --data {DATA} --out ./results

## 7. Report assets

Regenerates the LaTeX tables and the macros quoted in the text. `--keep_figures` protects the hand-polished figures from being overwritten.

In [ ]:
!mkdir -p report && cd report && python ../make_report_assets.py --results ../results --kaggle_public 0.79713 --kaggle_private 0.79295 --keep_figures

## 8. Package the outputs

In [ ]:
!zip -r project1_outputs.zip results report